# 25. Setor censitário e dependência espacial do deslocamento

Cada registro do CNEFE traz o código do setor censitário em que o IBGE o registrou. O setor é uma unidade espacial, e não um texto, e por isso permite conferir a consistência territorial dos pares sem depender da grafia de bairros ou da validade de CEPs. Ele também é a unidade natural para medir se o deslocamento se agrupa no espaço. Este notebook responde a cinco perguntas.

1. Que malha de setores corresponde aos códigos do CNEFE?
2. A coordenada do CNEFE cai dentro do setor que o próprio CNEFE declara?
3. Em que proporção dos pares o ponto do BHMap está num setor diferente do declarado pelo CNEFE, e como isso se compara com bairro e CEP?
4. O setor pode substituir o lote fiscal como geometria de comparação, ou como restrição do pareamento?
5. O deslocamento tem dependência espacial entre setores, e quanto ela depende da definição de vizinhança?

As entradas são as bases processadas do [NB01](01_ingestao.ipynb), a malha preliminar de setores do Censo 2022 publicada pelo IBGE, a camada de vilas e favelas da prefeitura e as saídas do [NB23](23_referencia_municipal_completa.ipynb): o pareamento por registro, as classes da cauda e os pares com lote único. As tabelas saem em `outputs/referencia_completa/tables` com o prefixo `nb25_`, e o agregado por setor, com a matriz de vizinhança, em `outputs/referencia_completa/geodados`.

In [1]:
from pathlib import Path
import sys, json, hashlib, importlib.metadata, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import matplotlib.pyplot as plt
from rapidfuzz import fuzz
from rapidfuzz.process import cpdist
from libpysal.weights import Queen, Rook, KNN
from esda import Moran, Moran_Local
from statsmodels.stats.multitest import multipletests
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/config.py').exists())
sys.path.insert(0, str(ROOT))
from src import config
from src.metrics import chave_identidade, separar_logradouro_numero, conciliar_logradouros
from src.normalize import normalize_logradouro
from src.setores import carregar_setores_preliminares
P = config.PROCESSED_DIR
OUT = ROOT/'outputs/referencia_completa'
T, F, G = OUT/'tables', OUT/'figures', OUT/'geodados'
for d in (T, F, G): d.mkdir(parents=True, exist_ok=True)
SEED, PERM = config.SEED, 9999

C = gpd.read_parquet(P/'cnefe_bh.parquet')
B = gpd.read_parquet(P/'bhmap_bh.parquet', columns=['IDEND', 'SIGLA_TIPO', 'NOME_LOGRA', 'NUMERO_IMO', 'NOME_BAIRR', 'NOME_REGIO', 'CEP', 'geometry'])
I = pd.read_parquet(P/'cnefe_identidade_completa.parquet')
S = carregar_setores_preliminares()
vila = gpd.read_parquet(P/'vila_favela.parquet')
assert C.crs == B.crs == S.crs == vila.crs and len(C) == 1180102 and len(B) == 757113
em_vila = pd.Series(shapely.within(C.geometry.to_numpy(), vila.geometry.union_all()), index=C.index)
apto = pd.to_numeric(C.COD_TIPO_ESPECI, errors='coerce').eq(103).fillna(False).astype(bool)
estab = pd.to_numeric(C.COD_ESPECIE, errors='coerce').isin([3, 4, 5, 6, 8]).fillna(False).astype(bool)
seg = pd.Series(np.select([apto.to_numpy(), estab.to_numpy()], ['apartamentos', 'estabelecimentos'], 'demais tipos'), index=C.index)
print(f'Setores da malha preliminar em Belo Horizonte: {len(S):,}')

Setores da malha preliminar em Belo Horizonte: 5,141


## 1. A malha que corresponde aos códigos do CNEFE

O IBGE publicou duas malhas de setores para o Censo 2022: a preliminar, usada na coleta, e a final. A camada de setores do GeoServer da prefeitura segue a malha final. Os códigos do CNEFE terminam com a letra P, de preliminar. A célula compara as duas malhas com os códigos do CNEFE.

In [2]:
final = gpd.read_parquet(config.SETOR_CENSITARIO_PARQUET, columns=['COD_SETOR', 'geometry'])
malhas = pd.DataFrame({
    'malha': ['preliminar do IBGE', 'final (camada da prefeitura)'],
    'setores': [len(S), len(final)],
    'registros_com_codigo_na_malha_pct': [100*C.COD_SETOR.isin(S.index).mean(), 100*C.COD_SETOR.str[:-1].isin(set(final.COD_SETOR)).mean()],
    'setores_do_cnefe_ausentes': [len(set(C.COD_SETOR) - set(S.index)), len(set(C.COD_SETOR.str[:-1]) - set(final.COD_SETOR))]})
assert malhas.loc[0, 'registros_com_codigo_na_malha_pct'] == 100
malhas.to_csv(T/'nb25_malhas_setores.csv', index=False)
print(malhas.round(2).to_string(index=False))

                       malha  setores  registros_com_codigo_na_malha_pct  setores_do_cnefe_ausentes
          preliminar do IBGE     5141                             100.00                          0
final (camada da prefeitura)     5166                              94.39                        302


Todos os códigos do CNEFE estão na malha preliminar, que tem 5.141 setores em Belo Horizonte. Na malha final, 302 setores do CNEFE não existem, e 5,6% dos registros ficam sem polígono. Além disso, o mesmo código pode descrever polígonos diferentes nas duas malhas. Por isso este notebook usa a malha preliminar.

## 2. A coordenada do CNEFE no próprio setor

O teste verifica se cada coordenada do CNEFE cai dentro do polígono do setor que o registro declara. Para as que caem fora, mede-se a distância até esse polígono. Os resultados saem por nível da coordenada, tipo de registro e vila.

In [3]:
proprio = S.geometry.reindex(C.COD_SETOR).to_numpy()
dentro = shapely.covers(proprio, C.geometry.to_numpy())
dist = np.where(dentro, 0.0, shapely.distance(proprio, C.geometry.to_numpy()))
pos = pd.DataFrame({'fora': ~dentro, 'dist_m': dist, 'nivel': C.NV_GEO_COORD.to_numpy(), 'tipo': seg.to_numpy(),
                    'vila': np.where(em_vila, 'em vila', 'fora de vila')}, index=C.index)
fora = pos[pos.fora]
resumo_pos = pd.Series({'registros': len(pos), 'fora_do_setor': len(fora), 'fora_pct': 100*pos.fora.mean(),
                        'distancia_mediana_m': fora.dist_m.median(), 'distancia_p90_m': fora.dist_m.quantile(.9),
                        'acima_10m': int(fora.dist_m.gt(10).sum()), 'acima_10m_pct': 100*fora.dist_m.gt(10).sum()/len(pos),
                        'acima_100m': int(fora.dist_m.gt(100).sum())})
def por(col):
    return pos.groupby(col).agg(registros=('fora', 'size'), fora_pct=('fora', lambda s: 100*s.mean()),
                                acima_10m_pct=('dist_m', lambda s: 100*s.gt(10).mean()), mediana_fora_m=('dist_m', lambda s: s[s > 0].median()))
tab_pos = pd.concat({c: por(c) for c in ['nivel', 'tipo', 'vila']})
resumo_pos.to_csv(T/'nb25_cnefe_no_proprio_setor.csv')
tab_pos.to_csv(T/'nb25_cnefe_no_proprio_setor_estratos.csv')
print(resumo_pos.round(2).to_string())
print(tab_pos.round(2).to_string())

registros              1180102.00
fora_do_setor           228638.00
fora_pct                    19.37
distancia_mediana_m          4.30
distancia_p90_m             17.78
acima_10m                51139.00
acima_10m_pct                4.33
acima_100m                2149.00
                        registros  fora_pct  acima_10m_pct  mediana_fora_m
nivel 1                    994242     19.14           4.23            4.21
      2                    173799     20.87           4.57            4.68
      3                      8306     24.58          13.44           11.50
      4                      3752      0.29           0.03            1.14
      6                         3      0.00           0.00             NaN
tipo  apartamentos         403623     23.66           7.81            5.97
      demais tipos         642844     15.90           2.00            3.23
      estabelecimentos     133635     23.16           5.09            4.55
vila  em vila              128149     17.85          

Um em cada cinco registros do CNEFE (228.638, 19,4%) tem a coordenada fora do polígono do próprio setor, mas quase sempre por poucos metros: a mediana da distância até o polígono é de 4,3 m e o P90 de 17,8 m. Só 4,3% dos registros ficam a mais de 10 m do seu setor, e 2.149 a mais de 100 m. A coleta registra a entrada do imóvel, junto à via, e quando o limite do setor corre pela via um ponto de fachada pode cair do outro lado dele sem que o registro esteja no setor errado. Os apartamentos e os estabelecimentos ficam fora com mais frequência (23,7% e 23,2%) que os demais tipos (15,9%), e as coordenadas posicionadas pela face de quadra quase nunca ficam fora. Como a posição exata da coordenada em relação ao limite é pouco informativa, a comparação dos pares usa o setor declarado pelo CNEFE.

## 3. Setor dos pares

O pareamento pela chave do [NB23](23_referencia_municipal_completa.ipynb) é reconstruído aqui para obter o ponto do BHMap de cada par. Cada ponto do BHMap recebe o setor que o contém. O par tem setor diferente quando esse setor não é o declarado pelo registro do CNEFE. Bairro e CEP são comparados como no NB23: bairros normalizados com token sort ratio de pelo menos 90, e CEPs válidos iguais.

A discordância principal verifica diretamente se o polígono declarado cobre a coordenada municipal, incluindo a borda. Uma segunda tabela compara os setores que contêm as duas coordenadas. Pares sem atribuição geométrica em um dos lados são separados do denominador. Se um ponto toca mais de um setor, considera-se concordância quando os conjuntos de setores têm interseção.

In [4]:
lc, nc = chave_identidade(*separar_logradouro_numero(C.LOGRAD_NUM))
lb, nb = chave_identidade(B.SIGLA_TIPO.fillna('')+' '+B.NOME_LOGRA.fillna(''), B.NUMERO_IMO)
mapa = conciliar_logradouros(lc.unique(), lb.unique(), 90, 2)
lc = lc.map(lambda x: mapa.get(x, x)).astype('string[python]')
nc, lb, nb = [s.astype('string[python]') for s in (nc, lb, nb)]
kc = (lc+'|'+nc).where(lc.notna() & lc.ne('') & nc.notna())
kb = (lb+'|'+nb).where(lb.notna() & lb.ne('') & nb.notna())
jb = gpd.sjoin(B[['geometry']], S.reset_index()[['CD_SETOR', 'geometry']], how='left', predicate='within')
setor_b = jb[~jb.index.duplicated()].CD_SETOR.reindex(B.index)
A = pd.DataFrame({'idx': C.index, 'chave': kc.to_numpy(), 'xc': C.geometry.x.to_numpy(), 'yc': C.geometry.y.to_numpy(), 'setor_c': C.COD_SETOR.to_numpy()})
Rb = pd.DataFrame({'idx_b': B.index, 'chave': kb.to_numpy(), 'xb': B.geometry.x.to_numpy(), 'yb': B.geometry.y.to_numpy(), 'setor_b': setor_b.to_numpy()})
J = A.dropna(subset=['chave']).merge(Rb.dropna(subset=['chave']), on='chave', how='inner')
J['dist'] = np.hypot(J.xc-J.xb, J.yc-J.yb)
base = J.loc[J.groupby('idx').dist.idxmin()].set_index('idx').sort_index()
assert len(base) == 1062044 and np.allclose(base.dist, I.dist_identidade.reindex(base.index), atol=1e-8)
# O setor declarado existe para todos os registros. Pontos na borda contam dentro.
geom_declarada = S.geometry.reindex(base.setor_c).to_numpy()
geom_b = B.geometry.reindex(base.idx_b).to_numpy()
base['setor_dif'] = ~shapely.covers(geom_declarada, geom_b)
# Diagnóstico adicional: comparar as duas coordenadas, sem usar o setor declarado.
# Intersects inclui bordas. Setores múltiplos não são arbitrariamente desempatados.
jc_geom = gpd.sjoin(C.loc[base.index, ['geometry']], S.reset_index()[['CD_SETOR', 'geometry']], how='left', predicate='intersects')
jb_geom = gpd.sjoin(B[['geometry']], S.reset_index()[['CD_SETOR', 'geometry']], how='left', predicate='intersects')
def conjuntos_setores(j):
    return j.dropna(subset=['CD_SETOR']).groupby(level=0).CD_SETOR.agg(frozenset)
sc_geom, sb_geom = conjuntos_setores(jc_geom), conjuntos_setores(jb_geom)
cg = sc_geom.reindex(base.index).map(lambda x: x if isinstance(x, frozenset) else frozenset())
bg = sb_geom.reindex(base.idx_b).map(lambda x: x if isinstance(x, frozenset) else frozenset())
validos_geom = np.array([bool(a) and bool(b) for a,b in zip(cg,bg)])
dif_geom = np.array([not bool(a & b) for a,b in zip(cg,bg)])
resumo_geom = pd.Series({'pares':len(base), 'pares_com_setor_nas_duas_coordenadas':int(validos_geom.sum()),
    'pares_sem_setor_em_alguma_coordenada':int((~validos_geom).sum()),
    'pares_em_setores_geometricos_distintos':int((validos_geom & dif_geom).sum()),
    'pct_distintos_entre_pares_com_duas_atribuicoes':100*dif_geom[validos_geom].mean(),
    'pares_com_atribuicao_multipla_na_borda':int(((cg.map(len).to_numpy()>1)|(bg.map(len).to_numpy()>1)).sum())})
resumo_geom.to_csv(T/'nb25_setores_das_duas_coordenadas.csv')
print(resumo_geom.round(3).to_string())

def bairro(s):
    return s.map(lambda v: normalize_logradouro(v) if pd.notna(v) else pd.NA).astype('string[python]').replace('', pd.NA)
def cep(s):
    x = s.astype('string[python]').str.replace(r'\.0$', '', regex=True).str.replace(r'\D', '', regex=True).str.zfill(8)
    return x.where(x.str.fullmatch(r'\d{8}', na=False) & x.ne('00000000'))
bc, bb = bairro(C.DSC_LOCALIDADE).reindex(base.index), bairro(B.NOME_BAIRR).reindex(base.idx_b)
ok = bc.notna().to_numpy() & bb.notna().to_numpy()
sim = np.full(len(base), np.nan)
sim[ok] = cpdist(bc.to_numpy()[ok].astype(str), bb.to_numpy()[ok].astype(str), scorer=fuzz.token_sort_ratio, workers=-1)
base['bairro_dif'] = np.where(ok, sim < 90, np.nan)
cc = cep(C.CEP).reindex(base.index).fillna('').to_numpy(dtype=str)
cb = cep(B.CEP).reindex(base.idx_b).fillna('').to_numpy(dtype=str)
base['cep_dif'] = np.where((cc != '') & (cb != ''), cc != cb, np.nan)
base['vila'] = em_vila.reindex(base.index).to_numpy()
base['lote_unico'] = base.index.isin(pd.read_parquet(OUT/'tables/nb23_ponto_lote_pares.parquet').index)
cauda = pd.read_parquet(OUT/'tables/nb23_cauda_adjudicada.parquet')
base['classe'] = cauda.veredito.reindex(base.index)
assert base.classe.notna().sum() == 29775 and base.setor_b.isna().sum() < 1000
faixa = pd.cut(base.dist, [-0.1, 10, 30, 100, 1000, np.inf], labels=['até 10 m', '10 a 30 m', '30 a 100 m', '100 m a 1 km', 'acima de 1 km'])
def discordancia(g):
    return g.agg(pares=('dist', 'size'), setor_dif_pct=('setor_dif', lambda s: 100*s.mean()),
                 bairro_dif_pct=('bairro_dif', lambda s: 100*s.mean()), cep_dif_pct=('cep_dif', lambda s: 100*s.mean()))
por_faixa = discordancia(base.groupby(faixa, observed=True))
por_faixa.loc['todos'] = [len(base), 100*base.setor_dif.mean(), 100*np.nanmean(base.bairro_dif), 100*np.nanmean(base.cep_dif)]
por_classe = discordancia(base[base.classe.notna()].groupby('classe')).sort_values('pares', ascending=False)
por_vila = discordancia(base.groupby(np.where(base.vila, 'em vila', 'fora de vila')))
por_faixa.to_csv(T/'nb25_setor_por_faixa.csv')
por_classe.to_csv(T/'nb25_setor_por_classe.csv')
por_vila.to_csv(T/'nb25_setor_por_vila.csv')
print(por_faixa.round(2).to_string()); print(por_classe.round(1).to_string()); print(por_vila.round(2).to_string())

pares                                             1062044.000
pares_com_setor_nas_duas_coordenadas              1061109.000
pares_sem_setor_em_alguma_coordenada                  935.000
pares_em_setores_geometricos_distintos             221313.000
pct_distintos_entre_pares_com_duas_atribuicoes         20.857
pares_com_atribuicao_multipla_na_borda                  0.000
                   pares  setor_dif_pct  bairro_dif_pct  cep_dif_pct
dist                                                                
até 10 m        539913.0           0.35            7.29         6.30
10 a 30 m       391803.0           0.83            7.25         8.84
30 a 100 m      100553.0           4.90            8.70        11.22
100 m a 1 km     21674.0          27.84           17.74        11.79
acima de 1 km     8101.0          99.94           98.99        92.91
todos          1062044.0           2.27            8.32         8.45
                                   pares  setor_dif_pct  bairro_dif_pct  cep

O ponto do BHMap está num setor diferente do declarado pelo CNEFE em 2,3% dos pares. A discordância cresce com a distância: 0,35% até 10 m, 0,8% de 10 a 30 m, 4,9% de 30 a 100 m, 27,8% de 100 m a 1 km e praticamente todos os pares acima de 1 km. Até 100 m, o setor discorda menos que o bairro, que diverge em cerca de 7% dos pares mais próximos por diferenças de grafia e de delimitação entre o nome declarado ao Censo e o bairro oficial. De 100 m a 1 km, o setor discorda mais (27,8% contra 17,7% do bairro), porque pares afastados atravessam limites de setor dentro do mesmo bairro. Acima de 1 km, os três atributos discordam em quase todos os pares.

Nas classes da cauda, o setor separa as chaves ambíguas, em que 94,8% dos pares estão em setores diferentes, dos imóveis extensos (28,0%) e do CNEFE fora do lote (24,9%), com o mesmo padrão do bairro e do CEP. Nas vilas e favelas, o setor discorda em 8,0% dos pares, contra 26,8% do bairro. O setor dá, portanto, uma leitura territorial mais estável que o bairro onde a toponímia é menos padronizada.

A comparação geométrica entre as duas coordenadas é distinta da comparação com o setor declarado. Entre 1.061.109 pares com setor atribuído nos dois lados, 221.313 (20,9%) caem em polígonos diferentes. Os 935 pares sem atribuição em pelo menos um lado ficam fora desse denominador. Não houve pontos com atribuição múltipla em borda. O resultado não contradiz os 2,3%: a primeira medida confronta dois pontos com a malha, enquanto a segunda confronta o ponto municipal com o código declarado pelo CNEFE.

## 4. Setor e lote como geometrias de comparação

O lote fiscal e o setor respondem a perguntas diferentes. O lote mede a diferença entre a entrada registrada pelo Censo e o ponto interno do cadastro dentro do mesmo imóvel, e só se aplica quando a chave liga um único lote. O setor, com centenas de metros, mede se os dois pontos estão na mesma porção do território. A tabela compara quantos pares cada geometria alcança.

In [5]:
alcance = pd.DataFrame({
    'pares': [len(base), int(base.vila.sum()), int((~base.vila).sum())],
    'com_lote_unico_pct': [100*base.lote_unico.mean(), 100*base[base.vila].lote_unico.mean(), 100*base[~base.vila].lote_unico.mean()],
    'com_setor_nos_dois_pontos_pct': [100*base.setor_b.notna().mean(), 100*base[base.vila].setor_b.notna().mean(), 100*base[~base.vila].setor_b.notna().mean()]},
    index=['todos', 'em vila', 'fora de vila'])
area = S.AREA_KM2.describe(percentiles=[.1, .5, .9])*1e6
alcance.to_csv(T/'nb25_alcance_lote_setor.csv')
print(alcance.round(2).to_string())
print(f"Área dos setores: mediana {area['50%']:,.0f} m², P10 {area['10%']:,.0f} m², P90 {area['90%']:,.0f} m²")

                pares  com_lote_unico_pct  com_setor_nos_dois_pontos_pct
todos         1062044               76.09                         100.00
em vila         92694               12.71                          99.98
fora de vila   969350               82.15                         100.00
Área dos setores: mediana 38,790 m², P10 12,087 m², P90 93,391 m²


O lote alcança 76,1% dos pares, e só 12,7% nas vilas e favelas, onde o cadastro fiscal é escasso. O setor alcança praticamente todos os pares, dentro e fora das vilas. Mas o setor mediano tem dezenas de milhares de metros quadrados, e uma distância até o setor seria zero para quase todos os pares, sem distinguir porta de ponto interno. O setor não substitui o lote como geometria de comparação. Ele complementa o lote como verificação territorial, e é a única disponível de forma homogênea nas vilas e favelas.

## 5. Pareamento restrito ao setor

Como no [NB18](18_restricao_territorial.ipynb) para bairro e CEP, todos os registros são pareados de novo exigindo que o ponto do BHMap esteja no setor declarado pelo CNEFE. O candidato continua sendo o mais próximo dentro da chave. Como o setor é uma unidade espacial, a restrição funciona como uma janela em torno do registro, e não como uma conferência textual independente.

In [6]:
Js = J[J.setor_c == J.setor_b]
rs = Js.loc[Js.groupby('idx').dist.idxmin()].set_index('idx')
mantidos = base.index.isin(rs.index)
comum = rs.index.intersection(base.index)
restricao = {'regra': 'logradouro, número e setor', 'pares': len(rs), 'cobertura_pct': 100*len(rs)/len(C),
             'mediana_m': rs.dist.median(), 'p90_m': rs.dist.quantile(.9),
             'candidato_alterado': int((rs.idx_b.reindex(comum) != base.idx_b.reindex(comum)).sum()),
             'cauda100_original': int(base.dist.gt(100).sum()), 'cauda100_final': int(rs.dist.gt(100).sum()),
             'cauda1000_original': int(base.dist.gt(1000).sum()), 'cauda1000_final': int(rs.dist.gt(1000).sum()),
             'registros_excluidos': int((~mantidos).sum())}
for nome, mask in [('em_vila', em_vila), ('fora_vila', ~em_vila)]:
    restricao[f'cobertura_{nome}_chave_pct'] = 100*base.index.isin(mask[mask].index).sum()/mask.sum()
    restricao[f'cobertura_{nome}_setor_pct'] = 100*rs.index.isin(mask[mask].index).sum()/mask.sum()
restricao = pd.Series(restricao)
territorial = pd.read_csv(OUT/'tables/nb23_restricao_territorial.csv')
assert restricao.cauda1000_original == 8101 and restricao.candidato_alterado <= restricao.pares
restricao.to_csv(T/'nb25_restricao_setor.csv')
print(restricao.to_string())
print(territorial[['regra', 'pares', 'cobertura_pct', 'cauda100_final', 'cauda1000_final']].round(2).to_string(index=False))

regra                            logradouro, número e setor
pares                                               1038425
cobertura_pct                                     87.994512
mediana_m                                          9.657192
p90_m                                             31.596993
candidato_alterado                                      537
cauda100_original                                     29775
cauda100_final                                        15719
cauda1000_original                                     8101
cauda1000_final                                           5
registros_excluidos                                   23619
cobertura_em_vila_chave_pct                       72.332987
cobertura_em_vila_setor_pct                       66.656782
cobertura_fora_vila_chave_pct                     92.147653
cobertura_fora_vila_setor_pct                     90.593876
            regra   pares  cobertura_pct  cauda100_final  cauda1000_final
logradouro_numero 1062044 

Exigir o mesmo setor mantém 88,0% dos registros pareados, contra 90,0% só pela chave e 82,4% com bairro. Os pares acima de 1 km caem de 8.101 para 5, e os acima de 100 m de 29.775 para 15.719. A cauda encolhe por exclusão, como com bairro e CEP, e os 537 registros que trocam de candidato ficam mais longe. A perda nas vilas e favelas continua maior que no resto da cidade (de 72,3% para 66,7%, contra 92,1% para 90,6%), mas é muito menor que a do bairro, que leva a cobertura das vilas a 52%. O setor elimina os pares distantes com menos exclusões porque não depende de como o bairro foi escrito, mas, sendo uma janela espacial, ele não indica qual candidato é o endereço certo.

## 6. Dependência espacial do deslocamento entre setores

O teste usa o I de Moran global sobre três medidas agregadas por setor declarado: a proporção de pares acima de 100 m, o logaritmo de 1 mais a mediana da distância e a proporção de pares com o ponto do BHMap em outro setor. Entram os setores com pelo menos 30 pares.

A vizinhança é definida de três formas. Queen, a principal, liga setores que compartilham aresta ou vértice. Rook exige aresta compartilhada. KNN8 liga cada setor aos oito centroides mais próximos, com a união simétrica das ligações. Todas as matrizes são padronizadas por linha. Setores sem vizinhos em Queen ou Rook saem das três matrizes, para comparar os mesmos setores.

A significância vem de 9.999 permutações, com p bilateral calculado das duas caudas da distribuição permutada. O z é calculado pela média e pelo desvio padrão das permutações, e também sob normalidade. A família de testes reúne medidas, matrizes e dois recortes, todos os pares e sem chaves ambíguas, e recebe correção de Holm.

O recorte sem chaves ambíguas usa o indicador do pareamento, definido pela diagonal maior que 100 m do retângulo envolvente dos pontos municipais. Ele não é a classe de evidência da cauda, que tem outras condições e depende da precedência. O mínimo de 30 pares e a exclusão de setores sem vizinhos são reaplicados a cada recorte. Restam 4.896 setores com todos os pares e 4.874 sem essas chaves, após a exclusão de dois e quatro setores, respectivamente.

In [7]:
def p_bilateral(observado, simulacoes):
    s = np.asarray(simulacoes)
    return min(1.0, 2*min((1+np.sum(s <= observado))/(len(s)+1), (1+np.sum(s >= observado))/(len(s)+1)))

base['ambigua'] = I.chave_ambigua.reindex(base.index).to_numpy()
np.random.seed(SEED)
linhas, agregados, descritivos = [], {}, {}
for recorte, d in [('todos os pares', base), ('sem chaves ambíguas', base[~base.ambigua])]:
    a = d.groupby('setor_c').agg(pares=('dist', 'size'), mediana_m=('dist', 'median'),
                                 cauda100=('dist', lambda v: v.gt(100).mean()), setor_dif=('setor_dif', 'mean'))
    geo = S.join(a, how='inner')
    geo = geo[geo.pares >= 30].reset_index()
    descritivos[recorte] = geo.copy()
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        q0, r0 = Queen.from_dataframe(geo, use_index=False), Rook.from_dataframe(geo, use_index=False)
    ilhas = sorted(set(q0.islands) | set(r0.islands))
    geo = geo.drop(index=ilhas).reset_index(drop=True)
    geo['log_mediana'] = np.log1p(geo.mediana_m)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        W = {'Queen': Queen.from_dataframe(geo, use_index=False), 'Rook': Rook.from_dataframe(geo, use_index=False),
             'KNN8': KNN.from_array(np.column_stack([geo.geometry.centroid.x, geo.geometry.centroid.y]), k=8).symmetrize()}
    for nome, w in W.items():
        assert not w.islands
        w.transform = 'r'
        for y in ['cauda100', 'log_mediana', 'setor_dif']:
            m = Moran(geo[y].to_numpy(), w, permutations=PERM)
            linhas.append({'recorte': recorte, 'matriz': nome, 'medida': y, 'setores': len(geo), 'ilhas_excluidas': len(ilhas),
                           'vizinhos_medio': w.mean_neighbors, 'I': m.I, 'E_I': m.EI, 'z_permutacao': m.z_sim,
                           'z_normal': m.z_norm, 'p_bilateral': p_bilateral(m.I, m.sim)})
    agregados[recorte] = (geo, W)
moran = pd.DataFrame(linhas)
moran['p_holm'] = multipletests(moran.p_bilateral, method='holm')[1]
moran.to_csv(T/'nb25_moran_setores.csv', index=False)
print(moran.round(4).to_string(index=False))

            recorte matriz      medida  setores  ilhas_excluidas  vizinhos_medio      I     E_I  z_permutacao  z_normal  p_bilateral  p_holm
     todos os pares  Queen    cauda100     4896                2          6.1111 0.2422 -0.0002       28.3582   28.5027       0.0002  0.0036
     todos os pares  Queen log_mediana     4896                2          6.1111 0.2950 -0.0002       34.9606   34.7071       0.0002  0.0036
     todos os pares  Queen   setor_dif     4896                2          6.1111 0.2215 -0.0002       25.7684   26.0679       0.0002  0.0036
     todos os pares   Rook    cauda100     4896                2          5.2112 0.2583 -0.0002       28.3028   28.1943       0.0002  0.0036
     todos os pares   Rook log_mediana     4896                2          5.2112 0.3062 -0.0002       33.4294   33.4077       0.0002  0.0036
     todos os pares   Rook   setor_dif     4896                2          5.2112 0.2294 -0.0002       25.0137   25.0395       0.0002  0.0036
     todos os

As três medidas têm dependência espacial positiva em todas as configurações. Com Queen e todos os pares, em 4.896 setores, o I de Moran é 0,24 para a proporção acima de 100 m (z de 28,4), 0,30 para o logaritmo da mediana (z de 35,0) e 0,22 para a proporção de pares em outro setor (z de 25,8). Com 9.999 permutações, o menor p alcançável é 0,0002, e todos os testes chegam a ele, com p de Holm de 0,004. Rook e KNN8 dão valores entre 0,18 e 0,31, e retirar as chaves ambíguas não muda a leitura. Setores com deslocamentos maiores ficam perto de setores com deslocamentos maiores, e a associação positiva permanece nas três definições de vizinhança testadas.

A intensidade relativa divide a proporção de pares acima de 100 m em cada setor pela proporção da cidade inteira. O mapa descreve todos os 4.898 setores com pelo menos 30 pares. Os dois setores sem vizinhos de contiguidade continuam no mapa e saem apenas das matrizes usadas na inferência, que retém 4.896 setores.

In [8]:
taxa_cidade = base.dist.gt(100).mean()
geo_i = descritivos['todos os pares'].copy()
geo_inferencia, _ = agregados['todos os pares']
geo_inferencia['razao'] = geo_inferencia.cauda100/taxa_cidade
geo_i['razao'] = geo_i.cauda100/taxa_cidade
distantes = geo_i.cauda100*geo_i.pares
alto = geo_i.razao.ge(2)
intensidade_setores = pd.Series({'taxa_cidade_pct': 100*taxa_cidade, 'setores': len(geo_i), 'pares_mapa': int(geo_i.pares.sum()), 'distantes_mapa': int(round((geo_i.cauda100*geo_i.pares).sum())),
                                 'distantes_razao_2_cidade_pct':100*(geo_i.cauda100*geo_i.pares)[geo_i.razao.ge(2)].sum()/base.dist.gt(100).sum(),
                                 'setores_sem_cauda_pct': 100*geo_i.cauda100.eq(0).mean(),
                                 'setores_razao_2': int(alto.sum()), 'setores_razao_2_pct': 100*alto.mean(),
                                 'pares_nos_setores_razao_2_pct': 100*geo_i.pares[alto].sum()/geo_i.pares.sum(),
                                 'distantes_nos_setores_razao_2_pct': 100*distantes[alto].sum()/distantes.sum()})
intensidade_setores.to_csv(T/'nb25_intensidade_setores.csv')
print(intensidade_setores.round(2).to_string())

taxa_cidade_pct                            2.80
setores                                 4898.00
pares_mapa                           1058994.00
distantes_mapa                         29460.00
distantes_razao_2_cidade_pct              78.39
setores_sem_cauda_pct                     49.76
setores_razao_2                          629.00
setores_razao_2_pct                       12.84
pares_nos_setores_razao_2_pct             10.77
distantes_nos_setores_razao_2_pct         79.23


In [9]:
geo, W = agregados['todos os pares']
from libpysal.io import open as ps_open
saida = geo.assign(id_gal=np.arange(len(geo)))[['id_gal', 'CD_SETOR', 'NM_SUBDIST', 'pares', 'mediana_m', 'log_mediana', 'cauda100', 'razao', 'setor_dif', 'geometry']]
saida.to_file(G/'nb25_setores_deslocamento.gpkg', layer='setores', driver='GPKG')
gal = ps_open(str(G/'nb25_setores_queen.gal'), 'w')
gal.write(W['Queen'])
gal.close()
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
por_faixa.drop(index='todos')[['setor_dif_pct', 'bairro_dif_pct', 'cep_dif_pct']].rename(
    columns={'setor_dif_pct': 'setor', 'bairro_dif_pct': 'bairro', 'cep_dif_pct': 'CEP'}).plot.bar(ax=axes[0], color=['#2b6c8f', '#c7843a', '#8a8a8a'], rot=0)
axes[0].set_xticks(range(5), ['até\n10 m', '10 a\n30 m', '30 a\n100 m', '100 m\na 1 km', 'acima\nde 1 km'])
S.plot(ax=axes[1], color='#e3e3e3', linewidth=0)
axes[0].set(ylabel='Pares com atributo diferente (%)', xlabel='Distância do par', title='Discordância territorial por distância')
from matplotlib.patches import Patch
classes = [('sem pares distantes', '#f4f7fb'), ('abaixo de 1', '#b9d3ea'), ('de 1 a 2', '#fdd0a2'), ('de 2 a 4', '#f16913'), ('4 ou mais', '#a50f15')]
classe = pd.cut(geo_i.razao, [-np.inf, 0, 1, 2, 4, np.inf], right=False, labels=[c for c, _ in classes]).astype(object)
classe[geo_i.razao.eq(0).to_numpy()] = 'sem pares distantes'
for nome, cor in classes:
    geo_i[classe.eq(nome).to_numpy()].plot(ax=axes[1], color=cor, linewidth=0)
axes[1].legend(handles=[Patch(color=c, label=n) for n, c in classes] + [Patch(color='#e3e3e3', label='menos de 30 pares')],
               title='Intensidade relativa', loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=8)
axes[1].set_axis_off()
axes[1].set_title('Intensidade relativa da cauda por setor')
fig.tight_layout()
fig.savefig(F/'nb25_setor_deslocamento.png', dpi=200, bbox_inches='tight')
plt.show()

25_setor_censitario.ipynb:cell_22:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


O mapa mantém todos os setores com pelo menos 30 pares, inclusive os dois excluídos da inferência por falta de vizinhos. As frações de pares e de pares distantes usam a população representada no mapa, e a tabela também informa a fração sobre todos os pares distantes da cidade. A camada `setores_mapa` do GeoPackage contém esse universo descritivo. A camada `setores` contém o universo inferencial e usa `id_gal` para se ligar à matriz Queen em formato GAL. A distribuição por setor é descritiva e não deve ser interpretada como um mapa de significância local.

## 7. Moran local e diagrama de dispersão

O I global resume a associação num único número. O diagrama de dispersão de Moran mostra de onde ele vem: cada setor aparece com o seu valor padronizado no eixo horizontal e com a média padronizada dos vizinhos, pela matriz Queen, no eixo vertical. A inclinação da reta ajustada é o próprio I. Os quadrantes alto-alto e baixo-baixo indicam vizinhos semelhantes, e os quadrantes alto-baixo e baixo-alto indicam vizinhos diferentes. O Moran local (LISA) decompõe o I por setor e testa cada um com 9.999 permutações condicionais, com p bilateral e ajustes de Benjamini e Hochberg (BH) e de Benjamini e Yekutieli (BY) para os 4.896 testes.

In [10]:
geo_q, W_q = agregados['todos os pares']
y = geo_q['cauda100'].to_numpy()
zy = (y - y.mean())/y.std()
wzy = W_q['Queen'].sparse @ zy
inclinacao = np.polyfit(zy, wzy, 1)[0]
I_queen = moran.query("recorte == 'todos os pares' and matriz == 'Queen' and medida == 'cauda100'").I.iloc[0]
assert np.isclose(inclinacao, I_queen)
lisa = Moran_Local(y, W_q['Queen'], permutations=PERM, keep_simulations=True, seed=SEED)
baixo = (1 + (lisa.rlisas <= lisa.Is[:, None]).sum(1))/(PERM + 1)
alto = (1 + (lisa.rlisas >= lisa.Is[:, None]).sum(1))/(PERM + 1)
geo_q['lisa_I'] = lisa.Is
geo_q['lisa_p'] = np.minimum(1, 2*np.minimum(baixo, alto))
geo_q['quadrante'] = pd.Series(lisa.q).map({1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}).to_numpy()
geo_q['sig_bh'] = multipletests(geo_q.lisa_p, method='fdr_bh')[0]
geo_q['sig_by'] = multipletests(geo_q.lisa_p, method='fdr_by')[0]
lisa_resumo = pd.DataFrame({'p_menor_que_005': geo_q[geo_q.lisa_p < 0.05].quadrante.value_counts(),
                            'BH': geo_q[geo_q.sig_bh].quadrante.value_counts(),
                            'BY': geo_q[geo_q.sig_by].quadrante.value_counts()}).fillna(0).astype(int)
lisa_resumo.loc['total'] = lisa_resumo.sum()
lisa_resumo.to_csv(T/'nb25_lisa_cauda.csv')
print(f'Inclinação do diagrama: {inclinacao:.4f} | I de Moran Queen: {I_queen:.4f}')
print(lisa_resumo.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
axes[0].scatter(zy, wzy, s=4, alpha=.35, color='#2b6c8f')
xs = np.linspace(zy.min(), zy.max(), 50)
axes[0].plot(xs, inclinacao*xs, color='#b2182b', lw=1.5, label=f'inclinação = I = {inclinacao:.2f}'.replace('.', ','))
axes[0].axhline(0, color='#888', lw=.7)
axes[0].axvline(0, color='#888', lw=.7)
for (tx, ty, rot) in [(.97, .97, 'HH'), (.03, .03, 'LL'), (.97, .03, 'HL'), (.03, .97, 'LH')]:
    axes[0].text(tx, ty, rot, transform=axes[0].transAxes, ha='right' if tx > .5 else 'left', va='top' if ty > .5 else 'bottom', fontsize=10, color='#555')
axes[0].set(xlabel='Proporção de pares acima de 100 m (padronizada)', ylabel='Média padronizada dos vizinhos (Queen)', title='Diagrama de dispersão de Moran')
axes[0].legend(loc='upper left', bbox_to_anchor=(0.08, 1.0), frameon=False)
cores = {'HH': '#b2182b', 'LL': '#2166ac', 'HL': '#ef8a62', 'LH': '#67a9cf'}
S.plot(ax=axes[1], color='#e3e3e3', linewidth=0)
geo_q[~geo_q.sig_bh].plot(ax=axes[1], color='#f7f7f7', linewidth=0)
for q, cor in cores.items():
    sel = geo_q[geo_q.sig_bh & geo_q.quadrante.eq(q)]
    if len(sel):
        sel.plot(ax=axes[1], color=cor, linewidth=0, label=f'{q} ({len(sel)})')
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color=c, label=f"{q} ({int(lisa_resumo.loc[q, 'BH']) if q in lisa_resumo.index else 0})") for q, c in cores.items()],
               title='Significativos (BH)', loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=8)
axes[1].set_axis_off()
axes[1].set_title('Moran local da proporção acima de 100 m')
fig.tight_layout()
fig.savefig(F/'nb25_moran_local.png', dpi=200, bbox_inches='tight')
plt.show()
colunas = ['id_gal', 'CD_SETOR', 'NM_SUBDIST', 'pares', 'mediana_m', 'log_mediana', 'cauda100', 'razao', 'setor_dif', 'lisa_I', 'lisa_p', 'quadrante', 'sig_bh', 'sig_by', 'geometry']
geo_q.assign(id_gal=np.arange(len(geo_q)))[colunas].to_file(G/'nb25_setores_deslocamento.gpkg', layer='setores', driver='GPKG')
# Mantenho no mapa os setores elegíveis sem vizinhos, excluídos apenas da inferência.
geo_i.to_file(G/'nb25_setores_deslocamento.gpkg', layer='setores_mapa', driver='GPKG')


Inclinação do diagrama: 0.2422 | I de Moran Queen: 0.2422
           p_menor_que_005   BH  BY
quadrante                          
HH                      88   19   0
HL                      18    6   0
LH                      73    9   0
LL                     349  154   0
total                  528  188   0
25_setor_censitario.ipynb:cell_25:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


No diagrama, a reta ajustada tem inclinação igual ao I de Moran Queen, 0,24. A maior parte dos setores fica perto da origem, com poucos pares distantes e vizinhos parecidos. Os setores do quadrante alto-alto são os que puxam a associação positiva. No Moran local, com o ajuste BH, 188 setores são significativos: 154 baixo-baixo, 19 alto-alto, 9 baixo-alto e 6 alto-baixo. Os alto-alto formam um agrupamento maior na borda nordeste do município e alguns setores isolados ao norte e a oeste. Os baixo-baixo são setores pequenos espalhados pela área central e pericentral, sem formar áreas contínuas. Com o ajuste BY, que não pressupõe independência entre os testes, nenhum setor fica significativo, porque 9.999 permutações dão p mínimo de 0,0002 para 4.896 testes. A dependência global é robusta, mas a delimitação de agrupamentos locais depende do ajuste escolhido.

## 8. Intensidade relativa no espaço contínuo

O mapa por setor depende de como os setores foram desenhados, e setores de tamanhos diferentes pesam de forma diferente. A densidade kernel verifica a mesma razão sem depender dessa divisão: onde os pares acima de 100 m se concentram no espaço contínuo. Como a densidade bruta só repetiria onde há mais endereços, a medida é relativa. A densidade dos pares distantes, dividida pelo seu total, é comparada com a densidade de todos os pares, dividida pelo total de pares. Valor 1 indica a mesma concentração dos pares em geral, e valor 2 indica o dobro. Os pontos são as coordenadas do CNEFE, contados em células de 100 m e suavizados por um kernel gaussiano com largura de banda em metros, no sistema SIRGAS 2000 / UTM 23S. A largura principal é de 500 m, com 250 e 1.000 m como sensibilidade. Ficam fora as células fora do município e as com densidade de pares abaixo do décimo percentil, onde a razão é instável.

In [11]:
from scipy.ndimage import gaussian_filter
CEL = 100
par_ok = I.pareado_identidade.to_numpy()
px, py = C.geometry.x.to_numpy()[par_ok], C.geometry.y.to_numpy()[par_ok]
distante = I.dist_identidade.to_numpy()[par_ok] > 100
x0, y0, x1, y1 = S.total_bounds
xe = np.arange(x0 - 2000, x1 + 2000 + CEL, CEL)
ye = np.arange(y0 - 2000, y1 + 2000 + CEL, CEL)
contagem_dist, _, _ = np.histogram2d(px[distante], py[distante], bins=[xe, ye])
contagem_todos, _, _ = np.histogram2d(px, py, bins=[xe, ye])
XX, YY = np.meshgrid((xe[:-1] + xe[1:])/2, (ye[:-1] + ye[1:])/2, indexing='ij')
no_municipio = shapely.contains_xy(S.geometry.union_all(), XX, YY)
na_vila = shapely.contains_xy(vila.geometry.union_all(), XX, YY)
linhas, grades = [], {}
for banda in [250, 500, 1000]:
    dd = gaussian_filter(contagem_dist, banda/CEL)
    dt = gaussian_filter(contagem_todos, banda/CEL)
    with np.errstate(divide='ignore', invalid='ignore'):
        razao = (dd/contagem_dist.sum())/(dt/contagem_todos.sum())
    valida = no_municipio & (dt > np.quantile(dt[no_municipio], 0.10))
    alta = valida & (razao >= 2)
    linhas.append({'banda_m': banda, 'celulas_validas': int(valida.sum()), 'razao_mediana': float(np.median(razao[valida])),
                   'razao_p95': float(np.quantile(razao[valida], .95)), 'area_razao_2_pct': 100*alta.sum()/valida.sum(),
                   'vila_na_area_valida_pct': 100*(valida & na_vila).sum()/valida.sum(),
                   'vila_na_area_razao_2_pct': 100*(alta & na_vila).sum()/max(alta.sum(), 1),
                   'pares_distantes_na_area_razao_2_pct': 100*dd[alta].sum()/dd[valida].sum()})
    grades[banda] = np.where(valida, razao, np.nan)
intensidade = pd.DataFrame(linhas).set_index('banda_m')
intensidade.to_csv(T/'nb25_intensidade_kernel.csv')
np.savez_compressed(G/'nb25_intensidade_relativa.npz', xe=xe, ye=ye, razao_500=grades[500], razao_250=grades[250], razao_1000=grades[1000])
print(intensidade.round(2).to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
from matplotlib.colors import LogNorm
for ax, banda in zip(axes, [250, 500, 1000]):
    im = ax.pcolormesh(xe, ye, grades[banda].T, cmap='RdBu_r', norm=LogNorm(vmin=0.25, vmax=4))
    vila.boundary.plot(ax=ax, color='k', linewidth=0.3)
    gpd.GeoSeries([S.geometry.union_all()], crs=S.crs).boundary.plot(ax=ax, color='#444', linewidth=0.6)
    ax.set_axis_off()
    ax.set_title(f'Largura de banda de {banda} m')
barra = fig.colorbar(im, ax=axes, shrink=.6, label='Intensidade relativa dos pares acima de 100 m')
barra.set_ticks([0.25, 0.5, 1, 2, 4])
barra.set_ticklabels(['0,25', '0,5', '1', '2', '4'])
fig.savefig(F/'nb25_intensidade_relativa.png', dpi=200, bbox_inches='tight')
plt.show()

         celulas_validas  razao_mediana  razao_p95  area_razao_2_pct  vila_na_area_valida_pct  vila_na_area_razao_2_pct  pares_distantes_na_area_razao_2_pct
banda_m                                                                                                                                                     
250                29807           0.59       5.28             17.24                     5.20                     11.66                                48.94
500                29807           0.76       3.39             15.63                     5.20                     11.61                                33.79
1000               29807           0.85       2.55              9.50                     5.15                      8.44                                17.76
25_setor_censitario.ipynb:cell_28:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


Com a largura de 500 m, a intensidade relativa mediana é de 0,76: na maior parte da cidade, os pares distantes são proporcionalmente menos frequentes que os pares em geral. As áreas em que eles são pelo menos duas vezes mais concentrados cobrem 15,6% da área analisada e reúnem um terço (33,8%) dos pares distantes. Elas ficam na periferia, a nordeste, a oeste, a sudoeste e ao norte, e a área central fica abaixo de 1. Os polígonos de vilas e favelas ocupam 11,6% dessas áreas, mais do dobro da sua participação na área analisada (5,2%), mas a maior parte das áreas de alta intensidade está fora deles. Com 250 m, o mapa fica mais fragmentado e as áreas acima de 2 chegam a 17,2%. Com 1.000 m, ficam 9,5%, nas mesmas regiões. Essa descrição complementar usa outras unidades e escalas e não demonstra ausência de efeito da agregação. Em linhas gerais, acompanha o mapa por setor: a discordância se concentra nas bordas do município, e não no centro.

## 9. Registro das entradas

O manifesto grava os hashes das entradas, a semente e as versões das bibliotecas.

In [12]:
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
entradas = [P/'cnefe_bh.parquet', P/'bhmap_bh.parquet', P/'cnefe_identidade_completa.parquet', P/'vila_favela.parquet', config.SETOR_PRELIMINAR_GPKG,
            OUT/'tables/nb23_cauda_adjudicada.parquet', OUT/'tables/nb23_ponto_lote_pares.parquet']
manifesto = {'notebook': '25_setor_censitario.ipynb', 'semente': SEED, 'permutacoes': PERM, 'minimo_pares_por_setor': 30,
             'malha': 'malha preliminar de setores do Censo 2022 (IBGE)', 'pares': int(len(base)),
             'versoes': {p: importlib.metadata.version(p) for p in ['numpy', 'pandas', 'geopandas', 'libpysal', 'esda', 'statsmodels']},
             'entradas_sha256': {str(p.relative_to(ROOT)): sha(p) for p in entradas}}
(T/'nb25_manifesto.json').write_text(json.dumps(manifesto, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(manifesto, ensure_ascii=False, indent=1))

{
 "notebook": "25_setor_censitario.ipynb",
 "semente": 42,
 "permutacoes": 9999,
 "minimo_pares_por_setor": 30,
 "malha": "malha preliminar de setores do Censo 2022 (IBGE)",
 "pares": 1062044,
 "versoes": {
  "numpy": "2.4.6",
  "pandas": "3.0.3",
  "geopandas": "1.1.3",
  "libpysal": "4.14.1",
  "esda": "2.8.2",
  "statsmodels": "0.14.6"
 },
 "entradas_sha256": {
  "data\\processed\\cnefe_bh.parquet": "b5f69c0c5c45bfcc76d015a7738952ba078035ca0eda8608acecc93d4af54692",
  "data\\processed\\bhmap_bh.parquet": "8c5917c24da7e165215660d6245eb6b07988e45263f83cccbc03b73a74a4cfc6",
  "data\\processed\\cnefe_identidade_completa.parquet": "daee4999c757131444e901f0dd6032dad4bc1aac6c39221af1c5df3ac84d5f6b",
  "data\\processed\\vila_favela.parquet": "f677a8b1c0996a08dacbcee253389a10d8a912b422e29d5592d19d0c047d2d91",
  "data\\raw\\ibge_setores_2022_preliminar\\MG_Malha_Preliminar_2022.gpkg": "5624c956bcd268874189194ef880bb05f43176b3520332a833dd6beda323fbeb",
  "outputs\\referencia_completa\\tables\

## Conclusão

Os códigos de setor do CNEFE seguem a malha preliminar do Censo 2022, e não a malha final que a prefeitura publica. Com a malha certa, todos os registros têm polígono. A coordenada do CNEFE cai fora do próprio setor em 19,4% dos registros, quase sempre a poucos metros do limite, como se espera de pontos de fachada junto às vias.

O setor é uma verificação territorial mais estável que o bairro e alcança todos os pares, inclusive nas vilas e favelas, onde o lote fiscal quase não chega. Ele não substitui o lote como geometria de comparação, porque não distingue a entrada do ponto interno do imóvel. Como restrição do pareamento, ele elimina os pares acima de 1 km com perda de cobertura bem menor que a do bairro, mas continua sendo exclusão, e não escolha do candidato certo.

O deslocamento tem dependência espacial positiva entre setores, com I de Moran entre 0,18 e 0,31 e z acima de 24 em todas as vizinhanças testadas. Os deslocamentos grandes se agrupam no território: um oitavo dos setores reúne quatro quintos dos pares distantes, e o teste não identifica a causa desse agrupamento. Localmente, 19 setores alto-alto se destacam com o ajuste BH, mas nenhum resiste ao ajuste BY. A densidade kernel relativa oferece uma descrição complementar em outras escalas: os pares distantes são mais concentrados na periferia que no centro.